# Unidad 2: Aprendizaje Automático 
## 🖼️ Visualización de Árboles de Decisión
### Exportar el árbol como imagen con Graphviz y Matplotlib
### Inteligencia Artificial - Lic. en Sistemas - FCAD/UNER

![Painting a tree](https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/notebooks/ml/images/pexels-ivaivo-14869517.jpg)

[Foto de Ivelin Donchev](https://www.pexels.com/es-es/foto/arte-pintura-pintando-cuadro-14869517/)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/ml/08_DecisionTree_View.ipynb)

## 🎯 ¿Qué vamos a aprender?

Además de entrenar un buen modelo, poder **visualizarlo** es una herramienta clave para:

- 🧑‍🏫 Explicar el modelo a personas sin conocimientos técnicos
- 🔍 Detectar decisiones incorrectas o ruido en los datos
- 📋 Documentar el razonamiento del clasificador

En este notebook vas a aprender dos formas de visualizar un Árbol de Decisión:

| Método | Librería | Ventaja |
|--------|----------|---------|
| `plot_tree()` | `matplotlib` | Sin instalación extra, inline en el notebook |
| `export_graphviz()` + `graphviz` | `graphviz` | Alta calidad, exporta a PDF/PNG |

---

> 📌 **Referencia:** [DataCamp - Decision Tree Classification in Python](https://www.datacamp.com/community/tutorials/decision-tree-classification-python)

## 📦 Paso 1: Importar Librerías

> ⚠️ **Graphviz requiere instalación especial.**  
> Además del paquete Python (`pip install graphviz`), necesitás el ejecutable del sistema:  
> - Linux: `sudo apt install graphviz`  
> - Mac: `brew install graphviz`  
> - Windows: descargar desde [graphviz.org](https://graphviz.org/download/)  
>
> Si no tenés graphviz instalado, las celdas con `matplotlib` funcionarán igual.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_graphviz
from sklearn.model_selection import train_test_split
from sklearn import metrics

# Intentamos importar graphviz (opcional)
try:
    import graphviz
    GRAPHVIZ_AVAILABLE = True
    print('✅ graphviz disponible — se usará para exportar el árbol')
except ImportError:
    GRAPHVIZ_AVAILABLE = False
    print('⚠️  graphviz no disponible — se usará matplotlib como alternativa')

print('✅ Resto de librerías importadas correctamente!')

## 📂 Paso 2: Cargar Datos y Entrenar el Modelo

In [ ]:
# 📥 Cargar el dataset
col_names = ['pregnant', 'glucose', 'bp', 'skin',
             'insulin', 'bmi', 'pedigree', 'age', 'label']

pima = pd.read_csv('https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/data/ml/pima-indians-diabetes.csv', header=0, names=col_names)

# 🎯 Features y Target
feature_cols = ['pregnant', 'insulin', 'bmi', 'age', 'glucose', 'bp', 'pedigree']
X = pima[feature_cols]
y = pima['label']

# ✂️ División 70/30
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=100
)

# 🌳 Entrenar el árbol (sin límite de profundidad)
model = DecisionTreeClassifier(random_state=100)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'✅ Modelo entrenado  |  Profundidad: {model.get_depth()}  |  Hojas: {model.get_n_leaves()}')
print(f'\n📊 Métricas (conjunto de prueba):')
print(f'  🎯 Accuracy : {metrics.accuracy_score(y_test, y_pred):.4f}')
print(f'  🔬 Precision: {metrics.precision_score(y_test, y_pred):.4f}')
print(f'  📡 Recall   : {metrics.recall_score(y_test, y_pred):.4f}')
print(f'  ⚖️  F1 Score : {metrics.f1_score(y_test, y_pred):.4f}')
print(f'  ✅ Score (model.score): {model.score(X_test, y_test):.4f}')

## 🎨 Paso 3: Visualizar con Matplotlib — `plot_tree()`

La forma más sencilla de visualizar un árbol directamente en el notebook es con `plot_tree()` de scikit-learn.

### Parámetros clave:

| Parámetro | Descripción |
|-----------|-------------|
| `feature_names` | Nombres de las columnas de entrada |
| `class_names` | Nombres de las clases de salida |
| `filled=True` | Colorea los nodos según la clase mayoritaria |
| `rounded=True` | Esquinas redondeadas (más estético) |
| `max_depth` | Niveles a mostrar (para árboles grandes) |
| `fontsize` | Tamaño de letra en el árbol |

In [ ]:
# 🎨 Visualización con Matplotlib (primeros 3 niveles)
fig, ax = plt.subplots(figsize=(22, 9))

plot_tree(
    model,
    feature_names=feature_cols,
    class_names=['No Diabetes', 'Diabetes'],
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=10,
    ax=ax
)

ax.set_title(
    f'Árbol de Decisión — primeros 3 niveles de {model.get_depth()}',
    fontsize=15, pad=20
)
plt.tight_layout()
plt.savefig('tree_matplotlib.png', dpi=150, bbox_inches='tight')
plt.show()

print('💾 Árbol guardado como tree_matplotlib.png')

## 🖼️ Paso 4: Visualizar con Graphviz — `export_graphviz()`

Graphviz genera árboles de **mayor calidad visual**, ideales para presentaciones y documentos.

### ¿Cómo funciona?

```
DecisionTreeClassifier  →  export_graphviz()  →  archivo .dot  →  graphviz.Source()  →  imagen PNG/PDF
```

El formato **DOT** es un lenguaje de descripción de grafos. `export_graphviz` convierte el árbol scikit-learn a este formato, y Graphviz lo renderiza como imagen.

> 💡 **Tip:** Para árboles muy grandes, conviene usar un árbol podado antes de visualizarlo con Graphviz.

In [ ]:
# Usamos un árbol podado para mejor legibilidad con Graphviz
model_vis = DecisionTreeClassifier(max_depth=4, random_state=100)
model_vis.fit(X_train, y_train)

# 📄 Generar el contenido DOT
dot_data = export_graphviz(
    model_vis,
    feature_names=feature_cols,
    class_names=['No Diabetes', 'Diabetes'],
    filled=True,
    rounded=True,
    special_characters=True,
    out_file=None           # None → retorna el string en lugar de escribir un archivo
)

print('📄 Primeras líneas del formato DOT generado:')
print(dot_data[:400] + '...')

In [ ]:
if GRAPHVIZ_AVAILABLE:
    # 🖼️ Renderizar con Graphviz
    graph = graphviz.Source(dot_data)

    # Mostrar inline en el notebook
    display(graph)

    # 💾 Guardar como PNG
    graph.render(filename='tree_graphviz', format='png', cleanup=True)
    print('\n💾 Árbol guardado como tree_graphviz.png')
else:
    print('⚠️  graphviz no disponible. Instalalo con:')
    print('   pip install graphviz')
    print('   sudo apt install graphviz  (Linux)')
    print('\n👇 Usando matplotlib como alternativa:')

    fig, ax = plt.subplots(figsize=(22, 10))
    plot_tree(
        model_vis,
        feature_names=feature_cols,
        class_names=['No Diabetes', 'Diabetes'],
        filled=True, rounded=True,
        fontsize=9, ax=ax
    )
    ax.set_title('🌳 Árbol Podado (max_depth=4) — via matplotlib', fontsize=14)
    plt.tight_layout()
    plt.savefig('tree_graphviz_alt.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('💾 Árbol guardado como tree_graphviz_alt.png')

## 🔎 Paso 5: Cómo Leer un Nodo del Árbol

Analicemos el nodo raíz (la primera pregunta que hace el árbol).

Cada nodo contiene la siguiente información:

```
┌──────────────────────────────┐
│   glucose <= 127.5           │  ← Condición de división (feature + umbral)
│   gini = 0.459               │  ← Impureza de Gini actual
│   samples = 537              │  ← Cuántas muestras pasan por este nodo
│   value = [348, 189]         │  ← [No Diabetes, Diabetes]
│   class = No Diabetes        │  ← Clase mayoritaria (predicción si fuera hoja)
└──────────────────────────────┘
```

### 📐 Impureza de Gini

$$\text{Gini} = 1 - \sum_{i=1}^{k} p_i^2$$

Donde $p_i$ es la proporción de la clase $i$ en el nodo.

| Valor Gini | Significado |
|-----------|-------------|
| **0.0** | Nodo puro: todas las muestras son de la misma clase ✅ |
| **0.5** | Nodo perfectamente mezclado (50% de cada clase) ⚠️ |

El árbol busca en cada paso la división que **minimiza la impureza de Gini** resultante.

In [ ]:
# 📐 Ilustración del cálculo de Gini para el nodo raíz
n_total = len(y_train)
n_diabetes    = y_train.sum()
n_no_diabetes = n_total - n_diabetes

p_diabetes    = n_diabetes / n_total
p_no_diabetes = n_no_diabetes / n_total
gini_raiz     = 1 - (p_diabetes**2 + p_no_diabetes**2)

print('📐 Cálculo de Gini para el conjunto de entrenamiento completo (nodo raíz):')
print(f'  Total muestras   : {n_total}')
print(f'  No Diabetes (0)  : {n_no_diabetes}  →  p₀ = {p_no_diabetes:.3f}')
print(f'  Diabetes    (1)  : {n_diabetes}  →  p₁ = {p_diabetes:.3f}')
print(f'\n  Gini = 1 - ({p_no_diabetes:.3f}² + {p_diabetes:.3f}²)')
print(f'  Gini = 1 - ({p_no_diabetes**2:.4f} + {p_diabetes**2:.4f})')
print(f'  Gini = {gini_raiz:.4f}')

print()
if gini_raiz > 0.4:
    print('  ⚠️  Gini alto → el nodo raíz tiene bastante mezcla de clases → hay trabajo por hacer!')
else:
    print('  ✅ Gini bajo → las clases están relativamente separadas desde el inicio.')

## 🏆 Paso 6: Importancia de las Features

Tanto el árbol completo como el podado pueden decirnos qué variables fueron más útiles para clasificar.

Comparamos la importancia entre el modelo completo y el podado.

In [ ]:
import numpy as np

imp_full   = model.feature_importances_
imp_pruned = model_vis.feature_importances_

x = np.arange(len(feature_cols))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, imp_full,   width, label='Sin podar',    color='steelblue',     edgecolor='black')
ax.bar(x + width/2, imp_pruned, width, label='Podado (d=4)', color='mediumseagreen', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(feature_cols, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Importancia relativa')
ax.set_title('Feature Importance: Árbol Completo vs Podado', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n🥇 Feature más importante (sin podar) : {feature_cols[imp_full.argmax()]}   ({imp_full.max():.3f})')
print(f'🥇 Feature más importante (podado d=4): {feature_cols[imp_pruned.argmax()]} ({imp_pruned.max():.3f})')

## 🗂️ Paso 7: Exportar el Árbol a una Imagen PNG

En muchos casos queremos guardar el árbol para incluirlo en un informe o presentación. Veamos las dos opciones:

In [ ]:
# 💾 Método 1: Guardar con matplotlib
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    model_vis,
    feature_names=feature_cols,
    class_names=['No Diabetes', 'Diabetes'],
    filled=True, rounded=True,
    fontsize=9, ax=ax
)
ax.set_title('Árbol de Decisión (max_depth=4)', fontsize=14)
plt.tight_layout()
plt.savefig('tree.png', dpi=200, bbox_inches='tight')
plt.show()
print('💾 Árbol guardado como tree.png (método matplotlib)')

# 💾 Método 2: Con Graphviz (si está disponible)
if GRAPHVIZ_AVAILABLE:
    dot = export_graphviz(
        model_vis,
        feature_names=feature_cols,
        class_names=['0', '1'],
        filled=True, rounded=True,
        special_characters=True
    )
    graphviz.Source(dot).render(filename='tree', format='png', cleanup=True)
    print('💾 Árbol guardado como tree.png (método graphviz)')
    display(Image('tree.png'))

## 🎓 Resumen y Conclusiones

### Lo que aprendimos:

1. 🎨 **`plot_tree()`** → visualización rápida e inline con `matplotlib`, sin dependencias externas
2. 🖼️ **`export_graphviz()` + `graphviz`** → visualizaciones de alta calidad exportables a PNG/PDF/SVG
3. 📐 **Impureza de Gini** → métrica que el árbol minimiza para encontrar las mejores divisiones
4. 🏆 **Feature Importance** → el árbol nos dice automáticamente qué variables son más relevantes

---

### 🔄 Comparación de métodos de visualización:

| | `plot_tree()` | `export_graphviz()` |
|-|--------------|--------------------|
| **Instalación** | Solo matplotlib ✅ | Requiere graphviz ⚙️ |
| **Calidad** | Básica 🟡 | Alta 🟢 |
| **Uso en notebook** | Inline ✅ | Requiere renderizado |
| **Exportar** | `savefig()` | `.render()` |
| **Formatos** | PNG, PDF, SVG | PNG, PDF, SVG, DOT |

---

### 🚀 ¿Qué sigue?

- ⚖️ **Comparar Árbol de Decisión vs Regresión Logística** en el dataset Titanic
- 🌲🌲🌲 **Random Forests**: muchos árboles en paralelo para mayor estabilidad
- 📊 **K-Fold Cross Validation**: evaluación más robusta que un único train/test split

---

> 📚 **Referencias:**  
> - [scikit-learn - Plot Tree](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html)  
> - [scikit-learn - Export Graphviz](https://scikit-learn.org/stable/modules/generated/sklearn.tree.export_graphviz.html)  
> - [Graphviz Documentation](https://graphviz.readthedocs.io/en/stable/)

---

© 2026 Cátedra Inteligencia Artificial — Lic. en Sistemas

Este notebook se distribuye bajo licencia [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).